# Event Stream and Firehose

The firehose is where ATProto becomes a continuously synchronized protocol participant —
events are ordered, sequenced, and replayable from cursors.

**What you'll learn:**
- Event sequencing with monotonically increasing sequence numbers
- Subscriber management (subscribe, unsubscribe, deliver)
- Cursor-based replay for late subscribers
- Broadcast fanout to all connected subscribers

**Estimated Time:** 20-25 minutes

## Step 1: Firehose Event

Every event on the firehose has a sequence number, a type, and a DID.
In real implementations, `FirehoseCommitEvent` carries CBOR-encoded commit data.

In [ ]:
@interface FirehoseEvent : NSObject
@property (nonatomic, assign) int seq;
@property (nonatomic, strong) NSString *type;
@property (nonatomic, strong) NSString *did;
@property (nonatomic, strong) NSDictionary *data;
- (NSString *)description;
@end

@implementation FirehoseEvent
- (NSString *)description {
    return [NSString stringWithFormat:@"#%d [%@] %@", _seq, _type, _did];
}
@end

## Step 2: Event Sequencer

The sequencer assigns monotonically increasing sequence numbers.
`SubscribeReposHandler.nextSequenceNumber` does the same in the real codebase.

In [ ]:
@interface EventSequencer : NSObject
@property (nonatomic, assign) int currentSeq;
- (int)nextSequenceNumber;
- (FirehoseEvent *)createEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data;
@end

@implementation EventSequencer
- (instancetype)init {
    self = [super init];
    if (self) { _currentSeq = 0; }
    return self;
}
- (int)nextSequenceNumber {
    _currentSeq = _currentSeq + 1;
    return _currentSeq;
}
- (FirehoseEvent *)createEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data {
    FirehoseEvent *event = [[FirehoseEvent alloc] init];
    event.seq = [self nextSequenceNumber];
    event.type = type;
    event.did = did;
    event.data = data;
    return event;
}
@end

EventSequencer *seq = [[EventSequencer alloc] init];
NSLog(@"Seq: %d", [seq nextSequenceNumber]);
NSLog(@"Seq: %d", [seq nextSequenceNumber]);
NSLog(@"Seq: %d", [seq nextSequenceNumber]);

## Step 3: Firehose Subscriber

Each subscriber tracks its own cursor (the last seq it received).
This mirrors `WebSocketConnection` state in the real codebase.

In [ ]:
@interface FirehoseSubscriber : NSObject
@property (nonatomic, strong) NSString *subscriberId;
@property (nonatomic, assign) int cursor;
@property (nonatomic, strong) NSMutableArray *eventBuffer;
- (void)receiveEvent:(FirehoseEvent *)event;
- (NSMutableArray *)eventsSinceCursor:(int)sinceCursor;
@end

@implementation FirehoseSubscriber
- (instancetype)initWithId:(NSString *)sid {
    self = [super init];
    if (self) {
        _subscriberId = sid;
        _cursor = 0;
        _eventBuffer = [NSMutableArray array];
    }
    return self;
}
- (void)receiveEvent:(FirehoseEvent *)event {
    [_eventBuffer addObject:event];
    _cursor = event.seq;
}
- (NSMutableArray *)eventsSinceCursor:(int)sinceCursor {
    NSMutableArray *results = [NSMutableArray array];
    for (int i = 0; i < [_eventBuffer count]; i++) {
        FirehoseEvent *e = [_eventBuffer objectAtIndex:i];
        if (e.seq > sinceCursor) {
            [results addObject:e];
        }
    }
    return results;
}
@end

## Step 4: Firehose

The firehose manages subscribers, broadcasts events, and supports cursor-based replay.
This mirrors `SubscribeReposHandler` which maintains `attachedConnections` and
broadcasts `FirehoseCommitEvent` objects.

In [ ]:
@interface Firehose : NSObject
@property (nonatomic, strong) NSMutableArray *subscribers;
@property (nonatomic, strong) NSMutableArray *eventLog;
@property (nonatomic, strong) EventSequencer *sequencer;
- (void)subscribe:(FirehoseSubscriber *)subscriber;
- (void)unsubscribe:(NSString *)subscriberId;
- (FirehoseEvent *)broadcastEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data;
- (void)replayFromCursor:(int)cursor toSubscriber:(FirehoseSubscriber *)subscriber;
@end

@implementation Firehose
- (instancetype)init {
    self = [super init];
    if (self) {
        _subscribers = [NSMutableArray array];
        _eventLog = [NSMutableArray array];
        _sequencer = [[EventSequencer alloc] init];
    }
    return self;
}
- (void)subscribe:(FirehoseSubscriber *)subscriber {
    [_subscribers addObject:subscriber];
    NSLog(@"Subscribed: %@", subscriber.subscriberId);
}
- (void)unsubscribe:(NSString *)subscriberId {
    for (int i = 0; i < [_subscribers count]; i++) {
        FirehoseSubscriber *s = [_subscribers objectAtIndex:i];
        if ([[s subscriberId] isEqualToString:subscriberId]) {
            [_subscribers removeObjectAtIndex:i];
            NSLog(@"Unsubscribed: %@", subscriberId);
            return;
        }
    }
}
- (FirehoseEvent *)broadcastEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data {
    FirehoseEvent *event = [_sequencer createEventWithType:type did:did data:data];
    [_eventLog addObject:event];
    // Deliver to all subscribers
    for (int i = 0; i < [_subscribers count]; i++) {
        FirehoseSubscriber *s = [_subscribers objectAtIndex:i];
        [s receiveEvent:event];
    }
    NSLog(@"Broadcast: %@", [event description]);
    return event;
}
- (void)replayFromCursor:(int)cursor toSubscriber:(FirehoseSubscriber *)subscriber {
    int count = 0;
    for (int i = 0; i < [_eventLog count]; i++) {
        FirehoseEvent *e = [_eventLog objectAtIndex:i];
        if (e.seq > cursor) {
            [subscriber receiveEvent:e];
            count = count + 1;
        }
    }
    NSLog(@"Replayed %d events from cursor %d to %@", count, cursor, subscriber.subscriberId);
}
@end

## Step 5: Full Firehose Demo

Subscribe two listeners, broadcast events, verify both receive them.
Unsubscribe one, broadcast more, verify only one receives.
Replay from cursor for a late subscriber.

In [ ]:
Firehose *fh = [[Firehose alloc] init];

// Subscribe two listeners
FirehoseSubscriber *sub1 = [[FirehoseSubscriber alloc] initWithId:@"relay-1"];
FirehoseSubscriber *sub2 = [[FirehoseSubscriber alloc] initWithId:@"appview-1"];
[fh subscribe:sub1];
[fh subscribe:sub2];

// Broadcast events
[fh broadcastEventWithType:@"commit" did:@"did:plc:abc" data:@{@"rev": @"1"}];
[fh broadcastEventWithType:@"commit" did:@"did:plc:def" data:@{@"rev": @"2"}];

NSLog(@"Sub1 cursor: %d, events: %d", sub1.cursor, [sub1.eventBuffer count]);
NSLog(@"Sub2 cursor: %d, events: %d", sub2.cursor, [sub2.eventBuffer count]);

// Unsubscribe sub2
[fh unsubscribe:@"appview-1"];

// Broadcast more
[fh broadcastEventWithType:@"identity" did:@"did:plc:abc" data:@{@"handle": @"alice.bsky.social"}];

NSLog(@"Sub1 events: %d", [sub1.eventBuffer count]);
NSLog(@"Sub2 events (unchanged): %d", [sub2.eventBuffer count]);

// Late subscriber joins and replays from cursor 0
FirehoseSubscriber *late = [[FirehoseSubscriber alloc] initWithId:@"relay-2"];
[fh subscribe:late];
[fh replayFromCursor:0 toSubscriber:late];
NSLog(@"Late subscriber events: %d", [late.eventBuffer count]);

## Exercise

Add a `broadcastIdentityChange:handle:` method to `Firehose` that creates an
identity-type event and broadcasts it. This mirrors `SubscribeReposHandler.broadcastIdentityChange:handle:`.

Hint: use `broadcastEventWithType:did:data:` with type `"identity"` and
data containing the handle.

In [ ]:
// Exercise: implement broadcastIdentityChange:handle:
// Your code here...

## Solution

In [ ]:
// Solution: broadcastIdentityChange wraps broadcastEventWithType
@interface Firehose2 : NSObject
@property (nonatomic, strong) NSMutableArray *subscribers;
@property (nonatomic, strong) NSMutableArray *eventLog;
@property (nonatomic, strong) EventSequencer *sequencer;
- (void)subscribe:(FirehoseSubscriber *)subscriber;
- (FirehoseEvent *)broadcastEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data;
- (void)broadcastIdentityChange:(NSString *)did handle:(NSString *)handle;
@end

@implementation Firehose2
- (instancetype)init {
    self = [super init];
    if (self) {
        _subscribers = [NSMutableArray array];
        _eventLog = [NSMutableArray array];
        _sequencer = [[EventSequencer alloc] init];
    }
    return self;
}
- (void)subscribe:(FirehoseSubscriber *)subscriber {
    [_subscribers addObject:subscriber];
}
- (FirehoseEvent *)broadcastEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data {
    FirehoseEvent *event = [_sequencer createEventWithType:type did:did data:data];
    [_eventLog addObject:event];
    for (int i = 0; i < [_subscribers count]; i++) {
        [[_subscribers objectAtIndex:i] receiveEvent:event];
    }
    return event;
}
- (void)broadcastIdentityChange:(NSString *)did handle:(NSString *)handle {
    [self broadcastEventWithType:@"identity" did:did data:@{@"handle": handle}];
    NSLog(@"Identity change: %@ -> %@", did, handle);
}
@end

// Test it
Firehose2 *fh2 = [[Firehose2 alloc] init];
FirehoseSubscriber *sub = [[FirehoseSubscriber alloc] initWithId:@"test"];
[fh2 subscribe:sub];
[fh2 broadcastIdentityChange:@"did:plc:abc" handle:@"alice.bsky.social"];
NSLog(@"Last event: %@", [[sub.eventBuffer lastObject] description]);

## Next Steps

You have learned about ATProto's core data structures and protocols. Continue to explore how these concepts apply when building a PDS.